In [2]:
# 왜 어떤 카테고리는 관심(View)은 높은데 실제 구매로 잘 이어지지 않는가? 에 대한 분석

In [4]:
# 05. 카테고리별 전환율 분석
# 상품 카테고리별 View → Cart → Purchase 전환 효율을 비교하고 저효율 카테고리를 찾음

In [6]:
import duckdb

parquet_path = r"..\data\processed\2019-*.parquet"

In [8]:
# category_code 유무에 따라 전체 이벤트와 구매 이벤트 비중 확인
# 카테고리명이 없는 데이터가 전체 행동과 구매에서 어느 정도 비중인지 확인
duckdb.sql(f"""
    SELECT
        CASE
            WHEN category_code IS NULL THEN 'missing_category'
            ELSE 'known_category'
        END AS category_status,

        COUNT(*) AS total_events,

        SUM(CASE WHEN event_type = 'view' THEN 1 ELSE 0 END) AS view_events,
        SUM(CASE WHEN event_type = 'cart' THEN 1 ELSE 0 END) AS cart_events,
        SUM(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) AS purchase_events

    FROM read_parquet('{parquet_path}')

    WHERE CAST(event_time AS DATE) <> '2019-11-15'

    GROUP BY category_status
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────────┬──────────────┬─────────────┬─────────────┬─────────────────┐
│ category_status  │ total_events │ view_events │ cart_events │ purchase_events │
│     varchar      │    int64     │   int128    │   int128    │     int128      │
├──────────────────┼──────────────┼─────────────┼─────────────┼─────────────────┤
│ missing_category │     33484264 │    32262388 │      814233 │          407643 │
│ known_category   │     70246063 │    66336010 │     2657908 │         1252145 │
└──────────────────┴──────────────┴─────────────┴─────────────┴─────────────────┘



In [ ]:
# 확인 결과
# - category_code 결측 이벤트가 약 3,348만 건으로 전체의 약 32%를 차지
# - 결측 데이터에도 purchase 이벤트가 약 40.8만 건 존재
# - 따라서 전체 분석에서 결측 데이터를 삭제하지 않고,
#   category_code가 필요한 카테고리 분석에서만 known_category를 사용

In [10]:
# category_code의 첫 번째 항목을 사용해 대분류별 이벤트 규모 확인
# electronics, appliances 등 대분류 카테고리별 행동 규모 확인
duckdb.sql(f"""
    SELECT
        SPLIT_PART(category_code, '.', 1) AS main_category,
        COUNT(*) AS total_events,

        SUM(CASE WHEN event_type = 'view' THEN 1 ELSE 0 END) AS views,
        SUM(CASE WHEN event_type = 'cart' THEN 1 ELSE 0 END) AS carts,
        SUM(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) AS purchases

    FROM read_parquet('{parquet_path}')

    WHERE category_code IS NOT NULL
      AND CAST(event_time AS DATE) <> '2019-11-15'

    GROUP BY main_category
    ORDER BY total_events DESC
""").show()

┌───────────────┬──────────────┬──────────┬─────────┬───────────┐
│ main_category │ total_events │  views   │  carts  │ purchases │
│    varchar    │    int64     │  int128  │ int128  │  int128   │
├───────────────┼──────────────┼──────────┼─────────┼───────────┤
│ electronics   │     38008027 │ 35137663 │ 1953697 │    916667 │
│ appliances    │     12506367 │ 11953583 │  378762 │    174022 │
│ computers     │      6088556 │  5902045 │  124179 │     62332 │
│ apparel       │      4295447 │  4226620 │   46610 │     22217 │
│ furniture     │      3146919 │  3090836 │   36240 │     19843 │
│ auto          │      2123340 │  2060291 │   41710 │     21339 │
│ construction  │      1724176 │  1667166 │   40510 │     16500 │
│ kids          │      1272130 │  1239793 │   20689 │     11648 │
│ accessories   │       594184 │   583340 │    7103 │      3741 │
│ sport         │       402084 │   393605 │    5754 │      2725 │
│ medicine      │        35216 │    32952 │    1610 │       654 │
│ country_

In [14]:
# 확인 결과
# - category_code가 존재하는 상품은 13개 대분류로 구성
# - electronics의 행동 규모가 가장 크고 appliances, computers 등이 뒤를 이음
# - 단순 이벤트 수는 인기/트래픽 규모를 의미할 뿐 전환 효율을 의미하지 않음
# - 다음 분석에서는 세션×상품 기준 전환율을 비교

In [16]:
# 세션×상품 기준으로 대분류별 View → Cart → Purchase 전환율 계산
# 세션×상품 기준으로 대분류별 View → Cart → Purchase 전환율 계산
duckdb.sql(f"""
    WITH event_times AS (
        SELECT
            user_session,
            product_id,

            MAX(category_code) AS category_code,

            MIN(CASE
                WHEN event_type = 'view'
                THEN event_time
            END) AS first_view_time,

            MIN(CASE
                WHEN event_type = 'cart'
                THEN event_time
            END) AS first_cart_time,

            MIN(CASE
                WHEN event_type = 'purchase'
                THEN event_time
            END) AS first_purchase_time

        FROM read_parquet('{parquet_path}')

        WHERE user_session IS NOT NULL
          AND CAST(event_time AS DATE) <> '2019-11-15'

        GROUP BY
            user_session,
            product_id
    ),

    funnel AS (
        SELECT
            SPLIT_PART(category_code, '.', 1) AS main_category,

            SUM(CASE
                WHEN first_view_time IS NOT NULL
                THEN 1 ELSE 0
            END) AS viewed,

            SUM(CASE
                WHEN first_view_time IS NOT NULL
                 AND first_cart_time IS NOT NULL
                 AND first_view_time <= first_cart_time
                THEN 1 ELSE 0
            END) AS carted,

            SUM(CASE
                WHEN first_view_time IS NOT NULL
                 AND first_cart_time IS NOT NULL
                 AND first_purchase_time IS NOT NULL
                 AND first_view_time <= first_cart_time
                 AND first_cart_time <= first_purchase_time
                THEN 1 ELSE 0
            END) AS purchased

        FROM event_times

        WHERE category_code IS NOT NULL

        GROUP BY main_category
    )

    SELECT
        main_category,
        viewed,
        carted,
        purchased,

        ROUND(
            carted * 100.0 / viewed,
            2
        ) AS view_to_cart_rate,

        ROUND(
            purchased * 100.0 / carted,
            2
        ) AS cart_to_purchase_rate,

        ROUND(
            purchased * 100.0 / viewed,
            2
        ) AS funnel_completion_rate

    FROM funnel

    ORDER BY viewed DESC
""").show()
duckdb.sql(f"""
    WITH event_times AS (
        SELECT
            user_session,
            product_id,

            MAX(category_code) AS category_code,

            MIN(CASE
                WHEN event_type = 'view'
                THEN event_time
            END) AS first_view_time,

            MIN(CASE
                WHEN event_type = 'cart'
                THEN event_time
            END) AS first_cart_time,

            MIN(CASE
                WHEN event_type = 'purchase'
                THEN event_time
            END) AS first_purchase_time

        FROM read_parquet('{parquet_path}')

        WHERE user_session IS NOT NULL
          AND CAST(event_time AS DATE) <> '2019-11-15'

        GROUP BY
            user_session,
            product_id
    ),

    funnel AS (
        SELECT
            SPLIT_PART(category_code, '.', 1) AS main_category,

            SUM(CASE
                WHEN first_view_time IS NOT NULL
                THEN 1 ELSE 0
            END) AS viewed,

            SUM(CASE
                WHEN first_view_time IS NOT NULL
                 AND first_cart_time IS NOT NULL
                 AND first_view_time <= first_cart_time
                THEN 1 ELSE 0
            END) AS carted,

            SUM(CASE
                WHEN first_view_time IS NOT NULL
                 AND first_cart_time IS NOT NULL
                 AND first_purchase_time IS NOT NULL
                 AND first_view_time <= first_cart_time
                 AND first_cart_time <= first_purchase_time
                THEN 1 ELSE 0
            END) AS purchased

        FROM event_times

        WHERE category_code IS NOT NULL

        GROUP BY main_category
    )

    SELECT
        main_category,
        viewed,
        carted,
        purchased,

        ROUND(
            carted * 100.0 / viewed,
            2
        ) AS view_to_cart_rate,

        ROUND(
            purchased * 100.0 / carted,
            2
        ) AS cart_to_purchase_rate,

        ROUND(
            purchased * 100.0 / viewed,
            2
        ) AS funnel_completion_rate

    FROM funnel

    ORDER BY viewed DESC
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────────┬──────────┬─────────┬───────────┬───────────────────┬───────────────────────┬────────────────────────┐
│ main_category │  viewed  │ carted  │ purchased │ view_to_cart_rate │ cart_to_purchase_rate │ funnel_completion_rate │
│    varchar    │  int128  │ int128  │  int128   │      double       │        double         │         double         │
├───────────────┼──────────┼─────────┼───────────┼───────────────────┼───────────────────────┼────────────────────────┤
│ electronics   │ 22938689 │ 1283501 │    616383 │               5.6 │                 48.02 │                   2.69 │
│ appliances    │  7230093 │  255520 │    103109 │              3.53 │                 40.35 │                   1.43 │
│ computers     │  3769749 │   89009 │     33961 │              2.36 │                 38.15 │                    0.9 │
│ apparel       │  3479664 │   33984 │     10720 │              0.98 │                 31.54 │                   0.31 │
│ furniture     │  2187340 │   26023 │  

In [18]:
# 확인 결과
# - electronics는 조회량이 가장 많으면서 최종 퍼널 완료율도 2.69%로 가장 높음
# - medicine은 완료율이 2.29%로 높지만 표본이 1.8만 view 정도로 매우 작음, 그래서 electronics와 동급으로 강조하기 힘듬
# - apparel은 조회량이 많지만 View → Cart 0.98%, 최종 완료율 0.31%로 매우 낮음
# - apparel은 높은 관심 대비 구매 전환 효율이 낮은 대표 카테고리 후보
# - furniture, accessories 등도 View → Cart 단계의 전환이 낮음
# - 카테고리별 차이는 특히 View → Cart 단계에서 크게 나타나는 경향이 있음

In [20]:
# 조회량이 충분한 대분류만 추려 관심 대비 구매전환 효율 비교
# 표본이 너무 작은 카테고리를 제외하고, 실제 주요 카테고리들의 전환 효율을 비교
duckdb.sql(f"""
    WITH event_times AS (
        SELECT
            user_session,
            product_id,
            MAX(category_code) AS category_code,
            MIN(CASE WHEN event_type = 'view' THEN event_time END) AS first_view_time,
            MIN(CASE WHEN event_type = 'cart' THEN event_time END) AS first_cart_time,
            MIN(CASE WHEN event_type = 'purchase' THEN event_time END) AS first_purchase_time
        FROM read_parquet('{parquet_path}')
        WHERE user_session IS NOT NULL
          AND CAST(event_time AS DATE) <> '2019-11-15'
        GROUP BY user_session, product_id
    ),
    funnel AS (
        SELECT
            SPLIT_PART(category_code, '.', 1) AS main_category,

            SUM(CASE
                WHEN first_view_time IS NOT NULL
                THEN 1 ELSE 0
            END) AS viewed,

            SUM(CASE
                WHEN first_view_time IS NOT NULL
                 AND first_cart_time IS NOT NULL
                 AND first_view_time <= first_cart_time
                THEN 1 ELSE 0
            END) AS carted,

            SUM(CASE
                WHEN first_view_time IS NOT NULL
                 AND first_cart_time IS NOT NULL
                 AND first_purchase_time IS NOT NULL
                 AND first_view_time <= first_cart_time
                 AND first_cart_time <= first_purchase_time
                THEN 1 ELSE 0
            END) AS purchased

        FROM event_times
        WHERE category_code IS NOT NULL
        GROUP BY main_category
    )

    SELECT
        main_category,
        viewed,

        ROUND(carted * 100.0 / viewed, 2) AS view_to_cart_rate,
        ROUND(purchased * 100.0 / carted, 2) AS cart_to_purchase_rate,
        ROUND(purchased * 100.0 / viewed, 2) AS funnel_completion_rate

    FROM funnel

    WHERE viewed >= 100000

    ORDER BY funnel_completion_rate DESC
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────────┬──────────┬───────────────────┬───────────────────────┬────────────────────────┐
│ main_category │  viewed  │ view_to_cart_rate │ cart_to_purchase_rate │ funnel_completion_rate │
│    varchar    │  int128  │      double       │        double         │         double         │
├───────────────┼──────────┼───────────────────┼───────────────────────┼────────────────────────┤
│ electronics   │ 22938689 │               5.6 │                 48.02 │                   2.69 │
│ appliances    │  7230093 │              3.53 │                 40.35 │                   1.43 │
│ construction  │  1039691 │              2.55 │                 36.17 │                   0.92 │
│ computers     │  3769749 │              2.36 │                 38.15 │                    0.9 │
│ auto          │  1313808 │              2.13 │                 39.69 │                   0.84 │
│ kids          │   894860 │              1.64 │                  35.2 │                   0.58 │
│ sport         │   

In [22]:
# 확인 결과
# - 조회량 10만 이상 주요 카테고리만 비교해도 전환 효율 차이가 큼
# - electronics는 높은 트래픽과 높은 전환율을 동시에 보임
# - apparel은 조회량이 약 348만으로 크지만 최종 완료율은 0.31%로 가장 낮음
# - furniture, accessories도 상대적으로 낮은 전환 효율을 보임
# - 주요 저효율 카테고리는 View → Cart 단계에서 특히 큰 손실이 발생

In [24]:
# apparel 내부의 세부 카테고리별 View → Cart → Purchase 전환율 비교
# 전환율이 낮았던 apparel 대분류 안에서 어떤 세부 카테고리가 실제 병목인지 확인
duckdb.sql(f"""
    WITH event_times AS (
        SELECT
            user_session,
            product_id,
            MAX(category_code) AS category_code,

            MIN(CASE WHEN event_type = 'view'
                THEN event_time END) AS first_view_time,

            MIN(CASE WHEN event_type = 'cart'
                THEN event_time END) AS first_cart_time,

            MIN(CASE WHEN event_type = 'purchase'
                THEN event_time END) AS first_purchase_time

        FROM read_parquet('{parquet_path}')

        WHERE user_session IS NOT NULL
          AND CAST(event_time AS DATE) <> '2019-11-15'

        GROUP BY user_session, product_id
    ),

    funnel AS (
        SELECT
            category_code,

            SUM(CASE
                WHEN first_view_time IS NOT NULL
                THEN 1 ELSE 0
            END) AS viewed,

            SUM(CASE
                WHEN first_view_time IS NOT NULL
                 AND first_cart_time IS NOT NULL
                 AND first_view_time <= first_cart_time
                THEN 1 ELSE 0
            END) AS carted,

            SUM(CASE
                WHEN first_view_time IS NOT NULL
                 AND first_cart_time IS NOT NULL
                 AND first_purchase_time IS NOT NULL
                 AND first_view_time <= first_cart_time
                 AND first_cart_time <= first_purchase_time
                THEN 1 ELSE 0
            END) AS purchased

        FROM event_times

        WHERE category_code LIKE 'apparel.%'

        GROUP BY category_code
    )

    SELECT
        category_code,
        viewed,
        carted,
        purchased,

        ROUND(carted * 100.0 / viewed, 2) AS view_to_cart_rate,
        ROUND(purchased * 100.0 / carted, 2) AS cart_to_purchase_rate,
        ROUND(purchased * 100.0 / viewed, 2) AS funnel_completion_rate

    FROM funnel

    WHERE viewed >= 10000

    ORDER BY viewed DESC
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────────────┬─────────┬────────┬───────────┬───────────────────┬───────────────────────┬────────────────────────┐
│      category_code      │ viewed  │ carted │ purchased │ view_to_cart_rate │ cart_to_purchase_rate │ funnel_completion_rate │
│         varchar         │ int128  │ int128 │  int128   │      double       │        double         │         double         │
├─────────────────────────┼─────────┼────────┼───────────┼───────────────────┼───────────────────────┼────────────────────────┤
│ apparel.shoes           │ 1931259 │  24481 │      7785 │              1.27 │                  31.8 │                    0.4 │
│ apparel.shoes.keds      │  702530 │   4850 │      1600 │              0.69 │                 32.99 │                   0.23 │
│ apparel.costume         │  312365 │   1918 │       635 │              0.61 │                 33.11 │                    0.2 │
│ apparel.dress           │  116974 │    330 │        81 │              0.28 │                 24.55 │  

In [26]:
# 확인 결과
# - apparel 내부에서도 다수의 세부 카테고리가 낮은 퍼널 완료율을 보임
# - apparel.shoes는 조회량이 약 193만으로 매우 크지만 완료율은 0.40%
# - dress, underwear, moccasins 등은 완료율이 0.1% 안팎으로 매우 낮음
# - apparel의 낮은 전환은 특정 한 세부 카테고리보다는 전반적인 View → Cart 약세와 관련된 것으로 보임
# - 단, 상품 상세정보·재고·할인 데이터가 없으므로 낮은 전환의 원인은 직접 확인할 수 없음

In [30]:
#--------------------------------------------------------------------------------------------------------------------------#

In [32]:
# Tableau용 주요 카테고리(너무 작은 카테고리는 제외)별 View → Cart → Purchase 전환율 데이터 생성
category_conversion = duckdb.sql(f"""
    WITH event_times AS (
        SELECT
            user_session,
            product_id,
            MAX(category_code) AS category_code,

            MIN(CASE
                WHEN event_type = 'view'
                THEN event_time
            END) AS first_view_time,

            MIN(CASE
                WHEN event_type = 'cart'
                THEN event_time
            END) AS first_cart_time,

            MIN(CASE
                WHEN event_type = 'purchase'
                THEN event_time
            END) AS first_purchase_time

        FROM read_parquet('{parquet_path}')

        WHERE user_session IS NOT NULL
          AND CAST(event_time AS DATE) <> '2019-11-15'

        GROUP BY
            user_session,
            product_id
    ),

    funnel AS (
        SELECT
            SPLIT_PART(category_code, '.', 1) AS main_category,

            SUM(CASE
                WHEN first_view_time IS NOT NULL
                THEN 1 ELSE 0
            END) AS viewed,

            SUM(CASE
                WHEN first_view_time IS NOT NULL
                 AND first_cart_time IS NOT NULL
                 AND first_view_time <= first_cart_time
                THEN 1 ELSE 0
            END) AS carted,

            SUM(CASE
                WHEN first_view_time IS NOT NULL
                 AND first_cart_time IS NOT NULL
                 AND first_purchase_time IS NOT NULL
                 AND first_view_time <= first_cart_time
                 AND first_cart_time <= first_purchase_time
                THEN 1 ELSE 0
            END) AS purchased

        FROM event_times

        WHERE category_code IS NOT NULL

        GROUP BY main_category
    )

    SELECT
        main_category,
        viewed,
        carted,
        purchased,

        ROUND(carted * 100.0 / viewed, 2) AS view_to_cart_rate,
        ROUND(purchased * 100.0 / carted, 2) AS cart_to_purchase_rate,
        ROUND(purchased * 100.0 / viewed, 2) AS funnel_completion_rate

    FROM funnel

    WHERE viewed >= 100000

    ORDER BY viewed DESC
""").df()

# Tableau용 CSV로 저장
category_conversion.to_csv(
    r"..\data\marts\dashboard_category_conversion.csv",
    index=False
)

category_conversion

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,main_category,viewed,carted,purchased,view_to_cart_rate,cart_to_purchase_rate,funnel_completion_rate
0,electronics,22938689.0,1283501.0,616383.0,5.60,48.02,2.69
1,appliances,7230093.0,255520.0,103109.0,3.53,40.35,1.43
2,computers,3769749.0,89009.0,33961.0,2.36,38.15,0.90
3,apparel,3479664.0,33984.0,10720.0,0.98,31.54,0.31
4,furniture,2187340.0,26023.0,8868.0,1.19,34.08,0.41
5,auto,1313808.0,27955.0,11094.0,2.13,39.69,0.84
6,construction,1039691.0,26501.0,9586.0,2.55,36.17,0.92
7,kids,894860.0,14680.0,5167.0,1.64,35.20,0.58
8,accessories,466886.0,5313.0,1790.0,1.14,33.69,0.38
9,sport,264919.0,4175.0,1390.0,1.58,33.29,0.52
